# 第 1 章 数学基础与线性系统 · 代码实战（Python）

> 本书章代码的 Python 实现，逐段中文注释；对应 MATLAB 版本见 `matlab/scanholo/ch1/`。

## 相关运算

`correlation.py`

In [ ]:
# ===== 书 1.2 例 1.6 · Table 1.5 correlation.m 的 Python 版 =====
# 频域实现二维相关（图案识别）：
#     f(x,y) (+) h(x,y) = IFFT{ F*(kx,ky) H(kx,ky) }        [书式 (1.2-7)(1.2-8)]
# 原书读两幅 smiley.bmp 做自相关（图 1.14 中心亮斑 = 两图"匹配"）。
# 这里用 numpy 现场画笑脸替代 imread；把 I2 换成别的图案即可复现图 1.15
# 的互相关（中心无亮斑）。
#
# MATLAB / Python 差异提示：
# 1) np.fft.fft2 / ifft2 / fftshift 与 MATLAB 同名函数约定一致，可逐行照抄。
# 2) 共轭放在哪个谱上（FI1.*conj(FI2) 还是 conj(FI1).*FI2）只决定输出是否
#    镜像翻转；对实图像取 abs 显示后看不出差别。数值验证见 verify_ch1.py。

import numpy as np
import matplotlib.pyplot as plt

# ----- 现场生成 256x256 笑脸位图（替代 I=imread('smiley.bmp','bmp')） -----
def make_smiley():
    N = 256
    yy, xx = np.mgrid[0:N, 0:N]                # 行 yy、列 xx（0..255）
    r = np.sqrt((xx - 128.0)**2 + (yy - 128.0)**2)
    img = np.zeros((N, N))
    img[np.abs(r - 90) < 4] = 1.0                                  # 外圈脸廓
    for ex in (93, 163):                                           # 两只眼睛
        img[(xx - ex)**2 + (yy - 88)**2 < 9**2] = 1.0
    ang = np.arctan2(yy - 118.0, xx - 128.0)                       # 嘴巴（下方圆弧）
    rm = np.sqrt((xx - 128.0)**2 + (yy - 118.0)**2)
    img[(np.abs(rm - 55) < 4) & (ang < -0.35) & (ang > -2.8)] = 1.0
    return img

# ----- 输入图像 1（参考图像）与输入图像 2（待识别图像） -----
I1 = make_smiley()        # I1=imread('smiley.bmp','bmp'); I1=I1(:,:,1)
I2 = make_smiley()        # 原书两幅同名文件 => 自相关；换成别的图案 => 互相关

fig, ax = plt.subplots(1, 3, figsize=(13, 4.6))
ax[0].imshow(I1, cmap='gray', origin='lower'); ax[0].set_title('Input image 1 (reference)'); ax[0].axis('off')
ax[1].imshow(I2, cmap='gray', origin='lower'); ax[1].set_title('Input image 2 (to recognize)'); ax[1].axis('off')

# ----- 频域计算相关 -----
FI1 = np.fft.fft2(I1)
FI1 = FI1 / np.abs(FI1).max()          # max2=max(max1); scale=1.0/max2; FI1=FI1.*scale
FI2 = np.fft.fft2(I2)
FI2 = FI2 / np.abs(FI2).max()

FPR = FI1 * np.conj(FI2)               # calculating correlation（书中 FPR=FI1.*conj(FI2)）
PR = np.fft.ifft2(FPR)
PR = np.fft.fftshift(PR)
PR = np.abs(PR) / np.abs(PR).max()     # 归一化后取模显示

ax[2].imshow(PR, cmap='gray', origin='lower')
ax[2].set_title('Correlation result (bright spot = match)')
ax[2].axis('off')

pk = np.unravel_index(np.argmax(PR), PR.shape)
print(f'相关峰位置 (row, col) = {pk}，应落在图像中心 (128, 128)')
plt.tight_layout()
plt.show()


## 二维FFT · 位图

`fft2d_bitmap.py`

In [ ]:
# ===== 书 1.1 例 1.2 · Table 1.4 fft2Dbitmap_image.m 的 Python 版 =====
# 对一幅 256x256 位图做二维傅里叶变换并显示幅度谱。
# 原书读 triangle.bmp（画图板随手画的三角形）；为了让你零依赖跑通，
# 这里用 numpy 现场画一个同样的三角形代替 imread。
#
# MATLAB / Python 差异提示：
# 1) np.fft.fft2 与 MATLAB fft2 同为"负指数"约定，可直接照抄；
#    fftshift 也同名同义（零频搬到中心）。
# 2) 原书 image(10*(abs(256*FI))) 是把幅度放大后按 0~255 灰度显示；
#    matplotlib 中等价写法是 imshow 归一化后的幅度谱。

import numpy as np
import matplotlib.pyplot as plt

# ----- 生成 256x256 三角形位图（替代 I=imread('triangle.bmp','bmp')） -----
N = 256
I = np.zeros((N, N))
yy, xx = np.mgrid[0:N, 0:N]
inside = (yy > 160) & (yy < 160 + (xx - 128) * 0.9) & (xx > 48) & (xx < 208)
I[inside] = 1.0                      # 白色三角形，黑色背景（幅度 1 = 白）

# ----- 书 figure(1)：显示输入图像 -----
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(I, cmap='gray', origin='lower')
ax[0].set_title('a) Bitmap image of a triangle')
ax[0].axis('off')

# ----- 计算傅里叶变换并中心化 -----
FI = np.fft.fft2(I)                  # I=I(:,:,1)：取一个通道（这里本来就是单通道）
FI = np.fft.fftshift(FI)
scale = 1.0 / np.abs(FI).max()       # max2=max(max1); scale=1.0/max2
FI = FI * scale

# ----- 书 figure(2)：显示变换的绝对值（对数拉伸便于观看星芒） -----
ax[1].imshow(20 * np.log10(np.abs(FI) + 1e-6), cmap='gray', origin='lower')
ax[1].set_title('b) |Transform| of a) (dB display)')
ax[1].axis('off')

plt.tight_layout()
plt.show()


## fft2drect

`fft2drect.py`

In [ ]:
# ===== 书 1.1 例 1.1 · Table 1.3 fft2Drect.m 的 Python 版 =====
# 二维 rect 函数（M x M 正方形）的傅里叶变换仿真，验证变换对
#     rect(x/a, y/b)  <=>  ab sinc(a*kx/2pi, b*ky/2pi)
# 书中运行时用 input 让你输 M（奇数 3~33），书中示例 M=11：
#     a = M/256 = 0.0429，第一零点 kx0 = 2*pi/a = 146.23 rad/(unit of length)
# 本版固定 M=11（想换值改这一个数即可）。
#
# MATLAB / Python 差异提示：
# 1) MATLAB 矩阵 1-based、Python 0-based：MATLAB 的第 128 行 = Python 下标 127。
# 2) 书中前向变换按约定是"正指数"，严格实现应为 ifft2(R)*N^2；
#    但 rect 是实偶函数，fft2(R)/N^2 与之完全相等，所以书里直接用 fft2。
#    numpy.fft.fft2 与 MATLAB fft2 指数约定一致（负指数），这里照抄书里写法。
# 3) MATLAB image() 的行号方向与 plt.imshow 不同，用 origin='lower' 对齐数学坐标。

import numpy as np
import matplotlib.pyplot as plt

# ----- 显示区域与采样 -----
L = 1.0                    # display area is L by L, L has unit of length
N = 256                    # number of sampling points
dx = L / (N - 1)           # dx : step size

# ----- 生成 M x M 正方形 rect(x/a, y/a)，M=奇数 -----
M = 11                     # 书中示例值：input('M (size of rect(x/a,y/a), enter odd numbers from 3-33-)=')
a = M / 256
kx0 = 2 * np.pi / a
print(f'a= {a:0.5g} [unit of length]')                    # 对应书中 sprintf
print(f'kx0 (first zero)= {kx0:0.5g} [radian/unit of length]')

R = np.zeros((256, 256))   # assign a matrix (256x256) of zeros
n = (M - 1) // 2
c = 128 - 1                # MATLAB 的第 128 个元素（1-based）= Python 下标 127
R[c - n:c + n + 1, c - n:c + n + 1] = 1.0   # 背景置 0、中心置 1 的正方形孔

# ----- 坐标轴标定（书中的写法：零频移到中央） -----
k = np.arange(256)
X = (k / 255) * 1.0 - L / 2        # X(k)=1/255*(k-1)-L/2，即 (k-1)*dx-L/2
Y = X.copy()
# Kx(k)=2*pi*(k-1)/((N-1)*dx) - pi/dx；本例 (N-1)*dx=1
Kx = 2 * np.pi * k / ((N - 1) * dx) - (2 * np.pi * (256 - 1)) / ((N - 1) * dx) / 2
Ky = Kx.copy()

fig, ax = plt.subplots(2, 3, figsize=(13, 8))

# ----- 书 figure(1)：rect 函数灰度图 -----
im0 = ax[0, 0].imshow(R, extent=[X[0] + dx / 2, X[-1] + dx / 2, Y[0] + dx / 2, Y[-1] + dx / 2],
                      cmap='gray', origin='lower')
ax[0, 0].set_title('rect function: gray-scale plot')
ax[0, 0].set_xlabel('x'); ax[0, 0].set_ylabel('y')

# ----- 书 figure(2)：rect 的中心横截面 -----
ax[0, 1].plot(X + dx / 2, R[:, 126])       # MATLAB R(:,127) -> Python 第 126 列
ax[0, 1].set_title('rect function: cross-section plot')
ax[0, 1].set_xlabel('x'); ax[0, 1].set_ylabel('rect(x/a)')
ax[0, 1].grid(True)
ax[0, 1].axis([-0.5, 0.5, -0.1, 1.2])

# ----- 计算傅里叶变换（书：FR=(1/256)^2*fft2(R) 后 fftshift） -----
FR = np.fft.fft2(R) / 256**2
FR = np.fft.fftshift(FR)

# ----- 书 figure(3)：|F| 沿 kx 的横截面（幅度 x10 显示） -----
ax[0, 2].plot(Kx - np.pi / (dx * (N - 1)), 10 * np.abs(FR[:, 126]))
ax[0, 2].set_title('Square-absolute value of Fourier transform\nof rect function: cross-section plot')
ax[0, 2].set_xlabel('kx'); ax[0, 2].set_ylabel('|a*b*sinc(a*kx/2pi)|')
ax[0, 2].axis([-800, 800, 0, np.abs(FR).max() * 10.1])
ax[0, 2].grid(True)

# ----- 书 figure(4)：|F|^2 三维网格图（matplotlib 用等高线图代替 mesh） -----
ax[1, 0].contourf(Kx, Ky, np.abs(FR)**2, levels=50, cmap='viridis')
ax[1, 0].set_title('Square-absolute value of FT: 3-D plot (contour), scale arbitrary')
ax[1, 0].set_xlabel('kx'); ax[1, 0].set_ylabel('ky')
ax[1, 0].set_aspect('equal')

# ----- 书 figure(5)：|F|^2 灰度图（gain=10000 强压动态范围） -----
gain = 10000
im1 = ax[1, 1].imshow(np.abs(FR)**2 / np.abs(FR).max()**2,
                      extent=[Kx[0], Kx[-1], Ky[0], Ky[-1]],
                      cmap='gray', origin='lower', vmax=255.0 / gain)
ax[1, 1].set_title('Square-absolute value of FT of the rect\nfunction: gray-scale plot')
ax[1, 1].set_xlabel('kx'); ax[1, 1].set_ylabel('ky')

ax[1, 2].axis('off')       # 占位空面板
plt.tight_layout()
plt.show()


## 菲涅耳 · 线性系统

`fresnel_lsi.py`

In [ ]:
# ===== 拓展 1.2.3 · 自由空间当作 LSI 系统：菲涅耳衍射的频域实现 =====
# 改编自你已有的 Fresnel Diffraction.m（规范化重写，物理参数与结果一致）：
# 矩形孔 f(x,y) 放在 z=0，求 z=5 cm 处的衍射光强。
# 本章视角（1.2 节）：自由空间是一个 LSI 系统，
#     路线 A（空间域卷积）：g = f * h，h 为菲涅耳脉冲响应；
#     路线 B（频域乘传递函数）：G = F*H —— 本脚本用的就是卷积定理的频域实现。
#
# MATLAB / Python 差异提示：
# 1) 离散卷积定理实现里，FFT 归一化因子 (1/N^2) 配 (N^2) 成对出现，互相抵消；
#    MATLAB 与 numpy 的 fft2/ifft2 约定一致，可逐行照抄。
# 2) meshgrid 默认 indexing='xy'：h(r,c) 中 MATLAB 的 X(r),Y(c) 对应
#    Python 里先对 X、Y 各取一列/一行组合，这里直接用广播 X[:,None]**2+Y[None,:]**2。

import numpy as np
import matplotlib.pyplot as plt

# ----- 1. 参数（与 Fresnel Diffraction.m 完全一致） -----
L = 1.0                    # 显示区域 L x L（单位 cm）
N = 256                    # 采样点数
dx = L / (N - 1)           # 采样步长 1/255 cm
M = 111                    # 正方形孔的像素宽度（111*dx ≈ 0.43 cm）
lam = 6.328e-5             # He-Ne 红光波长（cm），即 632.8 nm
z = 5.0                    # 传播距离（cm）

# ----- 2. 输入：中心 M x M 正方形孔 -----
R = np.zeros((N, N))
n = (M - 1) // 2
c = 128 - 1                # MATLAB 第 128 个元素 = Python 下标 127
R[c - n:c + n + 1, c - n:c + n + 1] = 1.0

# ----- 3. 坐标轴（与书中 fft2Drect.m 相同的标定方式） -----
k = np.arange(N)
X = k / (N - 1) - L / 2            # 单位 cm
Y = X.copy()

# ----- 4. 菲涅耳脉冲响应 h(x,y;z)（二次相位 + 高斯窗防混叠） -----
sigma = np.pi / (lam * z)                  # σ = pi/(lambda*z)，二次相位系数
r2 = X[:, None]**2 + Y[None, :]**2         # x^2+y^2 网格
h = 1j * (sigma / np.pi) * np.exp(-1j * sigma * r2) * np.exp(-800.0 * r2)
#   ↑ j*(σ/π)=j/(λz) 振幅因子；exp(-800*r2) 是高斯低通窗（原书 4*200=800），
#     用来把脉冲响应截断在显示窗口内，防止离散卷积的卷绕误差。

# ----- 5. 频域实现卷积（卷积定理：F{f*h} = F*H，照抄原书 MATLAB 流程） -----
FR = np.fft.fft2(R) / N**2          # 原书：FR=(1/256)^2*fft2(R)
H = np.fft.fft2(h) / N**2           # 原书：H=(1/256)^2*fft2(h)
g = np.fft.fftshift(np.fft.ifft2(FR * H)) * N**2
# 原书：hr=ifft2(HR); hr=(256^2)*hr; hr=fftshift(hr)
# 说明 1：本书前向变换为"正指数"，严格写法是 ifft2*N^2；卷积定理下两条路线
#         只差共轭/翻转，对本例的中心对称输入结果相同（数值验证见 verify_ch1.py）。
# 说明 2：这是"自然顺序"的循环卷积，奇数宽孔径中心在 MATLAB 第 128 个元素，
#         卷积峰会绕回后落在中心偏左约 2 像素处——原书代码同样如此，显示无碍。

I = np.abs(g)**2                            # 光强

# ----- 6. 可视化（对应原书 figure(1)(2)(3)(5)） -----
fig, ax = plt.subplots(2, 2, figsize=(11, 8.5))
ax[0, 0].imshow(R, extent=[X[0], X[-1], Y[0], Y[-1]], cmap='gray', origin='lower')
ax[0, 0].set_title('Input square aperture rect(x/a, y/a)')
ax[0, 0].set_xlabel('x (cm)'); ax[0, 0].set_ylabel('y (cm)')

ax[0, 1].plot(X + dx / 2, R[:, 126], '-o', ms=2)
ax[0, 1].set_title('rect function: cross-section plot')
ax[0, 1].set_xlabel('x (cm)'); ax[0, 1].axis([-0.5, 0.5, -0.1, 1.2]); ax[0, 1].grid(True)

ax[1, 0].plot(X + dx / 2, np.abs(g[:, 126]))
ax[1, 0].set_title('|g(x,y)| cross-section at z = 5 cm')
ax[1, 0].set_xlabel('x (cm)'); ax[1, 0].grid(True)

ax[1, 1].imshow(I, extent=[X[0], X[-1], Y[0], Y[-1]], cmap='gray', origin='lower')
ax[1, 1].set_title('2-D diffraction pattern (CCD view) at z = 5 cm')
ax[1, 1].set_xlabel('x (cm)'); ax[1, 1].set_ylabel('y (cm)')

plt.tight_layout()
plt.show()


## plot · sinc

`plot_sinc.py`

In [ ]:
# ===== 书 1.1 例 1.1 · Table 1.2 Plot_sinc.m 的 Python 版 =====
# 绘制 sinc(x) = sin(pi*x)/(pi*x) 曲线，验证其零点在 x = ±1, ±2, ±3, ...
#
# MATLAB / Python 差异提示：
# 1) MATLAB 的 x=-5.5:0.01:5.5 是闭区间等差序列；numpy.arange 不含终点，
#    这里用 np.linspace 精确复刻同样的 1101 个采样点。
# 2) MATLAB 中 0/0 得到 NaN 并在 plot 里留下一个针孔；numpy 中是 nan + RuntimeWarning。
#    原书代码没处理 x=0，本版在 x=0 处显式补上极限值 1（sinc 的定义值）。

import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-5.5, 5.5, 1101)          # 与书中 x=-5.5:0.01:5.5 相同的 1101 点
sinc = np.sin(np.pi * x) / (np.pi * x)    # 原书写法：sinc=sin(pi*x)./(pi*x)
sinc[x == 0] = 1.0                        # 修补 x=0 处的 0/0（原书未处理，见笔记"坑与陷阱"）

plt.plot(x, sinc)
plt.axis([-5.5, 5.5, -0.3, 1.1])
plt.grid(True)
plt.xlabel('x')
plt.ylabel('sinc (x)')
plt.title('sinc function')
plt.show()


## 数值验证 · ch1

`verify_ch1.py`

In [ ]:
# ===== 第 1 章数值验证脚本 =====
# 逐条验证笔记里的关键公式与书中数值，全部用 numpy 离散实现。
# 运行：python verify_ch1.py   （无绘图，纯文本结果）

import numpy as np

ok = lambda name, cond: print(f'[{"PASS" if cond else "FAIL"}] {name}')

# ---------- (a) sinc 定义与零点 ----------
x = np.array([1, 2, 3, 4, 5, -1, -2, -3], dtype=float)
ok('(a) sinc(x)=sin(pi x)/(pi x) 在 x=±1,±2,... 处为零',
   np.allclose(np.sin(np.pi * x) / (np.pi * x), 0, atol=1e-12))

# ---------- (b) 原书 Plot_sinc.m 的 0/0 陷阱 ----------
xs = -5.5 + 0.01 * np.arange(1101)          # 等价 MATLAB x=-5.5:0.01:5.5
hit0 = np.where(np.isclose(xs, 0.0, atol=1e-9))[0]
with np.errstate(invalid='ignore'):
    val0 = np.sin(np.pi * 0.0) / (np.pi * 0.0) if hit0.size == 0 else \
           np.sin(np.pi * xs[hit0[0]]) / (np.pi * xs[hit0[0]])
print(f'       网格是否精确命中 x=0: {hit0.size > 0}；x=0 处 sin(pi x)/(pi x) = {val0} (nan 即 0/0)')
ok('(b) 原书代码在 x=0 处产生 0/0（NaN），需要补极限值 1', np.isnan(val0) or hit0.size == 0)

# ---------- (c) 例 1.1 书中数值：M=11 -> a=0.0429, kx0=146.23 ----------
M = 11
a = M / 256
ok('(c) a = 11/256 = 0.0429（书中数值）', abs(a - 0.0429) < 5e-4)
ok('(c) 第一零点 kx0 = 2*pi/a = 146.23 rad/(unit length)（书中数值）',
   abs(2 * np.pi / a - 146.23) < 0.01)

# ---------- (d) 变换对 rect(x/a,y/b) <=> ab sinc(a kx/2pi, b ky/2pi) ----------
# 严格验证用"真实 DFT 频轴" K = 2*pi*n/(N*dx)（n=-N/2..N/2-1），
# 并把 rect 中心放在 fftshift 约定的网格中心（Python 下标 128）。
# 书中的 Kx 轴用 2*pi/((N-1)*dx) 步长，比真实 DFT 频轴大 N/(N-1) 倍（见"坑与陷阱"）。
N = 256
L = 1.0
dx = L / (N - 1)
a = M / 256
b = a
R = np.zeros((N, N)); n = (M - 1) // 2; c = 128
R[c - n:c + n + 1, c - n:c + n + 1] = 1.0
F = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(R)) * N**2)   # 书式(1.1-2a) 正指数
Kx = 2 * np.pi * np.fft.fftshift(np.fft.fftfreq(N, d=dx))
Ky = Kx.copy()
KX, KY = np.meshgrid(Kx, Ky)
F_ana = a * b * np.sinc(a * KX / (2 * np.pi)) * np.sinc(b * KY / (2 * np.pi))
err = np.abs(F_ana - F * dx**2).max() / np.abs(F_ana).max()     # DFT 乘 dx^2 逼近连续积分
print(f'       解析谱与 DFT 谱的最大相对偏差 = {err:.2e}')
print('       （偏差来源：显示窗口有限导致的频谱周期延拓混叠，'
      '以及书中 a=M/256 与实际采样宽度 M*dx=M/255 的半像素差，见"坑与陷阱"）')
ok('(d) rect 变换对：DFT(正指数) 与 ab*sinc 解析式一致（<5%）', err < 0.05)

# 第一零点位置（沿 kx，取中心行，找 |F| 的第一个极小值并与 2*pi/a 比较）
row = np.abs(F[128, :] * dx**2) / (np.abs(F * dx**2)).max()
band = (Kx > 2 * np.pi / a * 0.5) & (Kx < 2 * np.pi / a * 1.5)
i_min = np.argmin(np.where(band, row, np.inf))
dk = Kx[1] - Kx[0]
print(f'       测得第一零点 kx ≈ {Kx[i_min]:.2f}，理论 2*pi/a = {2*np.pi/a:.2f}（频格 {dk:.2f}）')
ok('(d) 频谱第一零点位于 2*pi/a（误差 < 1 个频格）', abs(Kx[i_min] - 2 * np.pi / a) < dk)

# ---------- (e) 卷积定理：频域乘 = 空间域卷积 ----------
rng = np.random.default_rng(0)
f1 = rng.random(16); h1 = rng.random(12)
g_direct = np.convolve(f1, h1)                        # 直接卷积，长度 N+M-1=27
g_fft = np.fft.ifft(np.fft.fft(f1, 27) * np.fft.fft(h1, 27))
err_e = np.abs(g_direct - g_fft.real).max()
print(f'       卷积输出尺寸 = {g_direct.size}（=16+12-1），直接卷积 vs FFT 最大偏差 = {err_e:.2e}')
ok('(e) 卷积定理：ifft(fft(f)*fft(h)) 与直接卷积一致，长度 N1+N2-1',
   g_direct.size == 27 and err_e < 1e-12)

# ---------- (f) 例 1.6：自相关中心最大 |C(0,0)| >= |C(x,y)| ----------
I1 = np.zeros((256, 256))
yy, xx = np.mgrid[0:256, 0:256]
r = np.sqrt((xx - 128.0)**2 + (yy - 128.0)**2)
I1[np.abs(r - 90) < 4] = 1.0
I1[(xx - 93)**2 + (yy - 88)**2 < 81] = 1.0
I1[(xx - 163)**2 + (yy - 88)**2 < 81] = 1.0
FI1 = np.fft.fft2(I1); FI1 = FI1 / np.abs(FI1).max()
PR = np.fft.ifft2(FI1 * np.conj(FI1))                 # 自相关（与 correlation.py 同一路线）
PR = np.fft.fftshift(PR)
C = np.abs(PR)
center = C[128, 128]
ok('(f) 自相关中心最大：|C(128,128)| >= max|C(x,y)|（书中式 1.2-6）',
   center >= C.max() - 1e-12 and abs(center - C.max()) < 1e-12)

# ---------- (g) 例 1.2.3 拓展：菲涅耳衍射图样的对称性 ----------
# 方孔与菲涅耳核都关于网格中心对称时，衍射光强应严格上下/左右镜像对称。
# 为让对称性严格成立，孔宽取偶数 112（几何中心落在两像素正中 127.5），
# 并用 ifftshift 把两个场都搬到"原点在 0"的自然 FFT 顺序里做循环卷积。
lam = 6.328e-5; z = 5.0
sigma = np.pi / (lam * z)
kq = np.arange(256); X = kq / 255 - 0.5; Y = X.copy()
Me = 112                                        # 偶数宽度，中心在 127.5
R2 = np.zeros((256, 256))
R2[128 - Me // 2:128 + Me // 2, 128 - Me // 2:128 + Me // 2] = 1.0
r2 = X[:, None]**2 + Y[None, :]**2              # 网格满足 X[255-r] = -X[r]，关于 127.5 对称
h = 1j * (sigma / np.pi) * np.exp(-1j * sigma * r2) * np.exp(-800.0 * r2)
g = np.fft.fftshift(np.fft.ifft2(
    np.fft.fft2(np.fft.ifftshift(R2)) * np.fft.fft2(np.fft.ifftshift(h)))) * 256**2
I = np.abs(g)**2
# 对称中心是整数下标 127（两个核都搬到"原点在 0"后，循环卷积中心落在
# 255.5+255.5 ≡ -1，经 fftshift 平移 +128 后为 127），故按 r -> 254-r 检验镜像。
J = I[:255, :255]
sym1 = np.abs(J - J[::-1, :]).max() / J.max()   # 上下镜像（关于第 127 行）
sym2 = np.abs(J - J[:, ::-1]).max() / J.max()   # 左右镜像（关于第 127 列）
pk = np.unravel_index(np.argmax(I), I.shape)
print(f'       强度上下镜像相对偏差 = {sym1:.2e}，左右镜像相对偏差 = {sym2:.2e}')
print(f'       光强峰位于 {pk}（孔边缘衍射条纹处，与镜像位置 {254-pk[0]} 成对），'
      f'Fresnel 数 a^2/(lambda*z) ≈ {(Me*dx/2)**2/(lam*z):.0f}')
ok('(g) 方孔菲涅耳衍射光强严格上下/左右镜像对称',
   sym1 < 1e-10 and sym2 < 1e-10)

print('\n全部验证完成。')
